- For custom Docker images for `accelerate config`, see the list [here](https://github.com/aws/deep-learning-containers/blob/master/available_images.md#huggingface-training-containers).<br>
- `accelerate` on Amazon SageMaker: [Click here](https://huggingface.co/docs/accelerate/main/en/usage_guides/sagemaker)
- To run the code: `accelerate launch ./src/training_app.py`

In [28]:
from src.data_connector import DataConnector


DATA_PATH = "dair-ai/emotion"
DATA_PERCENTAGE = 10

data = DataConnector.get_data(
    data_path=DATA_PATH,
    split_list=["train", "validation", "test"],
    split_perc=DATA_PERCENTAGE
)
data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
})

---

In [29]:
from src.model import Model


MODEL_ID = "gpt2"

model, tokenizer = Model.get_model(
    model_id=MODEL_ID,
    num_labels=len(data["train"].features["label"].names)
)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/mert/.virtualenvs/llm/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Mert: The cell above gave this warning:<be>
```
Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/{...}/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
```

In [30]:
model

GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=6, bias=False)
)

In [31]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>',
 'pad_token': '<|endoftext|>'}

---

In [32]:
from src.data_processing import DataProcessor


data_processor = DataProcessor(tokenizer=tokenizer)
tokenized_data = data_processor.transform(data=data)
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 200
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 200
    })
})

In [33]:
tokenized_data["test"]["labels"].type()

'torch.LongTensor'

---

In [34]:
import torch

In [37]:
d = torch.utils.data.DataLoader(
                dataset=tokenized_data["train"],
                batch_size=4,
                shuffle=True
            )

In [40]:
len(d.dataset)

1600